# 04 - Ensemble Learning : AdaBoost et XGBoost

## Women's E-Commerce Clothing Reviews

Les notebooks précédents ont permis de nettoyer les données (`01`), de construire le prétraitement
(`02`) et de comparer six modèles de classification (`03`). Le meilleur modèle, **Linear SVC**
(F1 = 0.9622), reste moins bon pour la classe minoritaire *non recommandé* (rappel ≈ 0.82).

Dans ce notebook, nous ajoutons deux méthodes d'**Ensemble Learning** de type **boosting** :

- **AdaBoost** : combine de nombreux arbres très simples, chaque arbre corrigeant les erreurs du précédent
  en donnant plus de poids aux exemples mal classés ;
- **XGBoost** : boosting de gradient optimisé, très performant sur les données tabulaires et textuelles hybrides.

Ces modèles sont comparés au Random Forest (*bagging*, déjà vu dans le notebook 03) et au Linear SVC
(modèle champion), avec **le même protocole** : même jeu de test, même prétraitement, `GridSearchCV`
avec `cv=6` et `scoring="f1"`.

L'accent est mis sur l'**analyse des erreurs**, en particulier sur la classe minoritaire.

**Plan du notebook :**

1. Préparation : données, split identique, prétraitement
2. Entraînement d'AdaBoost et de XGBoost
3. Évaluation et analyse d'erreurs
4. Export des résultats pour l'application Streamlit

## 1. Importation des bibliothèques

Nous importons les outils nécessaires et nous vérifions que **XGBoost** est bien installé
dans l'environnement du projet.

In [1]:
import warnings

import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

import xgboost

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

print("XGBoost version :", xgboost.__version__)

XGBoost version : 3.4.1


## 2. Chemins du projet

Comme dans le notebook 03, le dossier racine du projet est détecté automatiquement,
que le notebook soit lancé depuis `notebooks/` ou depuis la racine.

In [2]:
CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "data" / "processed").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "data" / "processed").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise FileNotFoundError("Le dossier data/processed est introuvable.")

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "reviews_clean.csv"
MODELS_DIR = PROJECT_ROOT / "models"

print("Project root :", PROJECT_ROOT)
print("Dataset      :", DATA_PATH.name, "->", "✓ existe" if DATA_PATH.exists() else "✗ introuvable")
print("Dossier models :", "✓ existe" if MODELS_DIR.exists() else "✗ introuvable")

Project root : c:\Users\ADNET\Desktop\projet fdd\Prediction-des-recommandations--dachats--de-vetements--pour--femmes
Dataset      : reviews_clean.csv -> ✓ existe
Dossier models : ✓ existe


## 3. Chargement du dataset nettoyé

Nous chargeons `reviews_clean.csv`, produit par le notebook 01.
L'option `keep_default_na=False` est la même que dans le notebook 02 : elle évite que
des chaînes comme `"Unknown"` ou un titre vide soient converties en valeurs manquantes.

In [3]:
df = pd.read_csv(DATA_PATH, keep_default_na=False)

print(f"Dimensions : {df.shape}")
print("Valeurs manquantes :", int(df.isna().sum().sum()))

Dimensions : (23465, 10)
Valeurs manquantes : 0


## 4. Reconstruction du  split train / test

Pour comparer équitablement les nouveaux modèles aux six modèles du notebook 03,
ils doivent être évalués sur **exactement les mêmes 4 693 avis de test**.

Nous refaisons donc le même découpage que dans le notebook 02 :
mêmes colonnes, `test_size=0.20`, `random_state=42` et `stratify=y`.

In [4]:
TARGET_COL = "Recommended IND"

X = df.drop(columns=[TARGET_COL, "Clothing ID"]).copy()
y = df[TARGET_COL].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"X_train : {X_train.shape} | X_test : {X_test.shape}")
print()
print("Proportion des classes (%) :")
print(pd.DataFrame({
    "Train": y_train.value_counts(normalize=True).sort_index() * 100,
    "Test": y_test.value_counts(normalize=True).sort_index() * 100
}).round(2))

X_train : (18772, 8) | X_test : (4693, 8)

Proportion des classes (%) :
                 Train   Test
Recommended IND              
0                17.78  17.77
1                82.22  82.23


## 5. Application du prétraitement sauvegardé

Le `ColumnTransformer` du notebook 02 a été sauvegardé dans `models/preprocessor.pkl`
après avoir été appris **sur le train uniquement**.

Ici, nous le chargeons et nous utilisons seulement `transform` (jamais `fit`) :
cela garantit les mêmes 13 035 variables et évite toute fuite de données.

In [5]:
preprocessor = joblib.load(MODELS_DIR / "preprocessor.pkl")

X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

y_train_arr = y_train.values
y_test_arr = y_test.values

print("X_train prétraité :", X_train_processed.shape)
print("X_test prétraité  :", X_test_processed.shape)
print("Type              :", type(X_train_processed).__name__)

X_train prétraité : (18772, 13035)
X_test prétraité  : (4693, 13035)
Type              : csr_matrix


## 6. Vérification de la reproductibilité

Avant d'entraîner de nouveaux modèles, nous vérifions que notre reconstruction est fidèle :
le modèle champion sauvegardé (`best_model.pkl`, un Linear SVC) doit retrouver sur ce jeu de test
les résultats du notebook 03 : **Accuracy = 0.9378** et **F1-Score = 0.9622**.

Si c'est le cas, les comparaisons entre les modèles sont valables.

In [6]:
champion = joblib.load(MODELS_DIR / "best_model.pkl")
y_pred_champion = champion.predict(X_test_processed)

acc = accuracy_score(y_test_arr, y_pred_champion)
f1 = f1_score(y_test_arr, y_pred_champion)

print(f"Modèle chargé : {type(champion).__name__}")
print(f"Accuracy : {acc:.4f} (attendu 0.9378)")
print(f"F1-Score : {f1:.4f} (attendu 0.9622)")

if round(acc, 4) == 0.9378 and round(f1, 4) == 0.9622:
    print("✓ Split et prétraitement identiques au notebook 03.")
else:
    print("⚠ Résultats différents : vérifier les données et le préprocesseur.")

Modèle chargé : LinearSVC
Accuracy : 0.9378 (attendu 0.9378)
F1-Score : 0.9622 (attendu 0.9622)
✓ Split et prétraitement identiques au notebook 03.


## 7. Protocole d'entraînement commun

Tous les modèles de ce notebook suivent le **même protocole que le notebook 03** :

- optimisation des hyperparamètres par `GridSearchCV` ;
- validation croisée stratifiée à **6 plis** (`cv=6`) ;
- métrique d'optimisation : **F1-Score** (classe 1, *recommandé*) ;
- évaluation finale sur le **jeu de test** de 4 693 avis, jamais utilisé pendant l'optimisation.

Le paramètre `QUICK_TEST` permet de vérifier rapidement que tout fonctionne (grilles minuscules,
aucune sauvegarde). **Il doit être à `False` pour l'entraînement final.**

In [12]:
import time

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier

RANDOM_STATE = 42
CV_FOLDS = 6
QUICK_TEST = False   

if QUICK_TEST:
    print(" MODE TEST RAPIDE ACTIVÉ ")

results = []        
trained_models = {} 
cv_tables = []      


def summarize(name, grid):
    """Évalue le meilleur modèle d'un GridSearchCV sur le jeu de test et mémorise les résultats."""
    model = grid.best_estimator_
    y_pred = model.predict(X_test_processed)
    y_proba = model.predict_proba(X_test_processed)[:, 1]

    row = {
        "Model": name,
        "Best Params": grid.best_params_,
        "CV F1 (6 plis)": round(grid.best_score_, 4),
        "Accuracy": round(accuracy_score(y_test_arr, y_pred), 4),
        "F1-Score": round(f1_score(y_test_arr, y_pred), 4),
        "ROC-AUC": round(roc_auc_score(y_test_arr, y_proba), 4),
    }
    results.append(row)
    trained_models[name] = model

    table = pd.DataFrame(grid.cv_results_)[
        ["params", "mean_test_score", "std_test_score", "rank_test_score", "mean_fit_time"]
    ].copy()
    table.insert(0, "Model", name)
    table["params"] = table["params"].astype(str)
    cv_tables.append(table)

    print(f"=== {name} ===")
    print("Meilleurs hyperparamètres :", grid.best_params_)
    print(f"CV F1 : {row['CV F1 (6 plis)']} | Test -> Accuracy : {row['Accuracy']} | "
          f"F1 : {row['F1-Score']} | ROC-AUC : {row['ROC-AUC']}\n")
    print(classification_report(
        y_test_arr, y_pred,
        target_names=["Non recommandé (0)", "Recommandé (1)"], digits=3
    ))

## 8. AdaBoost

**AdaBoost** (*Adaptive Boosting*) construit une suite d'arbres très simples, appelés
*weak learners* (apprenants faibles). Après chaque arbre, les avis **mal classés reçoivent plus de poids** :
l'arbre suivant se concentre donc sur les cas difficiles. La prédiction finale est un vote pondéré
de tous les arbres.

**Hyperparamètres optimisés :**

- `n_estimators` : nombre d'arbres ;
- `learning_rate` : poids donné à chaque arbre (petite valeur = apprentissage plus prudent) ;
- `estimator__max_depth` : profondeur des arbres faibles (1 = *stump*, un seul test par arbre).

In [8]:
if QUICK_TEST:
    param_grid_ada = {
        "n_estimators": [20],
        "learning_rate": [0.5],
        "estimator__max_depth": [1],
    }
else:
    param_grid_ada = {
        "n_estimators": [50, 100, 200],
        "learning_rate": [0.5, 1.0],
        "estimator__max_depth": [1, 2],
    }

n_fits = np.prod([len(v) for v in param_grid_ada.values()]) * CV_FOLDS
print(f"AdaBoost : {n_fits} entraînements à effectuer (combinaisons x {CV_FOLDS} plis)")

grid_ada = GridSearchCV(
    AdaBoostClassifier(
        estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
        random_state=RANDOM_STATE
    ),
    param_grid_ada,
    cv=CV_FOLDS,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)

t0 = time.perf_counter()
grid_ada.fit(X_train_processed, y_train_arr)
print(f"Durée : {(time.perf_counter() - t0) / 60:.1f} min\n")

summarize("AdaBoost", grid_ada)

AdaBoost : 72 entraînements à effectuer (combinaisons x 6 plis)
Fitting 6 folds for each of 12 candidates, totalling 72 fits
Durée : 8.1 min

=== AdaBoost ===
Meilleurs hyperparamètres : {'estimator__max_depth': 2, 'learning_rate': 1.0, 'n_estimators': 200}
CV F1 : 0.9639 | Test -> Accuracy : 0.9339 | F1 : 0.9593 | ROC-AUC : 0.9743

                    precision    recall  f1-score   support

Non recommandé (0)      0.781     0.874     0.825       834
    Recommandé (1)      0.972     0.947     0.959      3859

          accuracy                          0.934      4693
         macro avg      0.876     0.910     0.892      4693
      weighted avg      0.938     0.934     0.935      4693



## 9. XGBoost

**XGBoost** (*Extreme Gradient Boosting*) construit aussi les arbres l'un après l'autre,
mais chaque nouvel arbre corrige l'**erreur résiduelle** (le gradient de la fonction de perte)
des arbres précédents. Il ajoute une régularisation et gère très bien les matrices creuses,
ce qui convient à nos 13 035 variables TF-IDF + catégorielles.

**Hyperparamètres optimisés :**

- `n_estimators` : nombre d'arbres ;
- `learning_rate` : taux d'apprentissage (contribution de chaque arbre) ;
- `max_depth` : profondeur maximale des arbres.

XGBoost utilise déjà tous les cœurs du processeur pour construire chaque arbre
(`n_jobs=-1`) ; `GridSearchCV` lance donc les combinaisons l'une après l'autre.

In [9]:
if QUICK_TEST:
    param_grid_xgb = {
        "n_estimators": [20],
        "learning_rate": [0.3],
        "max_depth": [3],
    }
else:
    param_grid_xgb = {
        "n_estimators": [100, 200, 300],
        "learning_rate": [0.05, 0.1, 0.5],
        "max_depth": [3, 5],
    }

n_fits = np.prod([len(v) for v in param_grid_xgb.values()]) * CV_FOLDS
print(f"XGBoost : {n_fits} entraînements à effectuer (combinaisons x {CV_FOLDS} plis)")

grid_xgb = GridSearchCV(
    XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    param_grid_xgb,
    cv=CV_FOLDS,
    scoring="f1",
    n_jobs=1,
    verbose=1
)

t0 = time.perf_counter()
grid_xgb.fit(X_train_processed, y_train_arr)
print(f"Durée : {(time.perf_counter() - t0) / 60:.1f} min\n")

summarize("XGBoost", grid_xgb)

XGBoost : 108 entraînements à effectuer (combinaisons x 6 plis)
Fitting 6 folds for each of 18 candidates, totalling 108 fits
Durée : 41.6 min

=== XGBoost ===
Meilleurs hyperparamètres : {'learning_rate': 0.5, 'max_depth': 3, 'n_estimators': 100}
CV F1 : 0.9654 | Test -> Accuracy : 0.9339 | F1 : 0.9596 | ROC-AUC : 0.9733

                    precision    recall  f1-score   support

Non recommandé (0)      0.799     0.839     0.819       834
    Recommandé (1)      0.965     0.954     0.960      3859

          accuracy                          0.934      4693
         macro avg      0.882     0.897     0.889      4693
      weighted avg      0.935     0.934     0.935      4693



## 10. Référence : Random Forest (bagging)

Pour comparer **boosting** et **bagging**, nous réentraînons le Random Forest avec les
hyperparamètres retenus dans le notebook 03 (`n_estimators=200`, `max_depth=None`,
`min_samples_split=2`), sans refaire la recherche. Ce modèle n'est pas sauvegardé
(il est volumineux) : il sert uniquement à l'évaluation de ce notebook.

In [10]:
t0 = time.perf_counter()
rf_ref = RandomForestClassifier(
    n_estimators=20 if QUICK_TEST else 200,
    max_depth=None,
    min_samples_split=2,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_ref.fit(X_train_processed, y_train_arr)
print(f"Durée : {(time.perf_counter() - t0) / 60:.1f} min")

y_pred_rf = rf_ref.predict(X_test_processed)
print(f"Random Forest (référence) -> Accuracy : {accuracy_score(y_test_arr, y_pred_rf):.4f} | "
      f"F1 : {f1_score(y_test_arr, y_pred_rf):.4f} | "
      f"ROC-AUC : {roc_auc_score(y_test_arr, rf_ref.predict_proba(X_test_processed)[:, 1]):.4f}")

Durée : 0.2 min
Random Forest (référence) -> Accuracy : 0.9342 | F1 : 0.9604 | ROC-AUC : 0.9723


## 11. Sauvegarde des modèles et des résultats de la recherche

Nous sauvegardons dans `models/` :

- `adaboost_model.pkl` et `xgboost_model.pkl` : les deux modèles optimisés ;
- `ensemble_gridsearch_results.csv` : le détail de toutes les combinaisons testées,
  utile pour formaliser les hyperparamètres retenus dans le rapport.



In [11]:
summary_df = pd.DataFrame(results)
print(summary_df.to_string(index=False))

if QUICK_TEST:
    print("\n⚠ Mode test rapide : aucune sauvegarde effectuée.")
else:
    joblib.dump(trained_models["AdaBoost"], MODELS_DIR / "adaboost_model.pkl")
    joblib.dump(trained_models["XGBoost"], MODELS_DIR / "xgboost_model.pkl")
    pd.concat(cv_tables, ignore_index=True).to_csv(
        MODELS_DIR / "ensemble_gridsearch_results.csv", index=False
    )
    print("\n✓ Sauvegardés : adaboost_model.pkl, xgboost_model.pkl, ensemble_gridsearch_results.csv")

   Model                                                            Best Params  CV F1 (6 plis)  Accuracy  F1-Score  ROC-AUC
AdaBoost {'estimator__max_depth': 2, 'learning_rate': 1.0, 'n_estimators': 200}          0.9639    0.9339    0.9593   0.9743
 XGBoost            {'learning_rate': 0.5, 'max_depth': 3, 'n_estimators': 100}          0.9654    0.9339    0.9596   0.9733

✓ Sauvegardés : adaboost_model.pkl, xgboost_model.pkl, ensemble_gridsearch_results.csv
